# Credit Card Fraud Detection — XGBoost + SMOTE

Complete Google Colab implementation for the IEEE-CIS Fraud Detection case study.

**Case study requirements covered:** XGBoost, severe class imbalance, SMOTE, threshold tuning, model evaluation, feature importance and SHAP interpretation.

In [ ]:
# Install dependencies
!pip -q install xgboost imbalanced-learn pyarrow shap

In [ ]:
# Imports and project folders

import os
import gc
import json
import time
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve,
    precision_recall_curve, precision_score,
    recall_score, f1_score
)

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

os.makedirs("/content/data", exist_ok=True)
os.makedirs("/content/figs", exist_ok=True)

print("Environment ready.")

In [ ]:
# Upload your own IEEE-CIS ZIP file

uploaded = files.upload()

zip_files = [name for name in uploaded if name.lower().endswith(".zip")]

if not zip_files:
    raise FileNotFoundError("Please upload the IEEE-CIS dataset ZIP file.")

zip_path = os.path.join("/content", zip_files[0])
print("Uploaded:", zip_path)

In [ ]:
# Extract ZIP and locate the required CSV files

extract_path = "/content/ieee_fraud_data"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

required = {}

for root, _, filenames in os.walk(extract_path):
    for filename in filenames:
        if filename in {
            "train_transaction.csv",
            "train_identity.csv",
            "test_transaction.csv",
            "test_identity.csv"
        }:
            required[filename] = os.path.join(root, filename)

print("Files found:")
for name, path in required.items():
    print(f"{name}: {path}")

if "train_transaction.csv" not in required:
    raise FileNotFoundError("train_transaction.csv not found.")

if "train_identity.csv" not in required:
    raise FileNotFoundError("train_identity.csv not found.")

In [ ]:
# Load and merge training data

t0 = time.time()

train_transaction = pd.read_csv(required["train_transaction.csv"])
train_identity = pd.read_csv(required["train_identity.csv"])

print("Transaction data:", train_transaction.shape)
print("Identity data:", train_identity.shape)

train = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

del train_transaction, train_identity
gc.collect()

print("Merged data:", train.shape)
print("Loading/merging time:", round(time.time() - t0, 2), "seconds")

In [ ]:
# Class imbalance analysis

counts = train["isFraud"].value_counts().sort_index()
percent = train["isFraud"].value_counts(normalize=True).sort_index() * 100

print("Legitimate:", int(counts.get(0, 0)))
print("Fraudulent:", int(counts.get(1, 0)))
print("Fraud rate:", round(percent.get(1, 0), 4), "%")

plt.figure(figsize=(7, 5))
bars = plt.bar(
    ["Legitimate", "Fraudulent"],
    [counts.get(0, 0), counts.get(1, 0)]
)

plt.title("Class Distribution")
plt.xlabel("Transaction Class")
plt.ylabel("Number of Transactions")

for bar in bars:
    value = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        value,
        f"{int(value):,}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.savefig("/content/figs/01_class_imbalance.png", dpi=120)
plt.show()

In [ ]:
# Feature engineering and preprocessing

# Remove columns with more than 90% missing values.
missing_fraction = train.isna().mean()

high_missing_cols = [
    c for c in missing_fraction[missing_fraction > 0.90].index
    if c not in ["TransactionID", "TransactionDT", "isFraud"]
]

train.drop(columns=high_missing_cols, inplace=True)

# Time features.
train["Transaction_hour"] = (
    (train["TransactionDT"] // 3600) % 24
).astype("int8")

train["Transaction_day"] = (
    (train["TransactionDT"] // (3600 * 24)) % 7
).astype("int8")

# Transaction amount features.
train["TransactionAmt_log"] = np.log1p(
    train["TransactionAmt"].astype("float32")
).astype("float32")

train["TransactionAmt_decimal"] = (
    (
        train["TransactionAmt"]
        - train["TransactionAmt"].fillna(0).astype("int32")
    ) * 1000
).astype("float32")

# Frequency encoding.
for col in ["card1", "card2", "addr1", "P_emaildomain"]:
    if col in train.columns:
        freq = train[col].value_counts(dropna=False)
        train[f"{col}_freq"] = train[col].map(freq).astype("float32")

# Convert categorical columns to integer codes.
categorical_columns = train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in categorical_columns:
    train[col] = (
        train[col]
        .astype("category")
        .cat.codes
        .astype("int32")
    )

# Replace infinities.
train.replace([np.inf, -np.inf], np.nan, inplace=True)

# Median imputation for numeric columns.
numeric_columns = train.select_dtypes(
    include=[np.number]
).columns.tolist()

numeric_columns = [
    c for c in numeric_columns
    if c != "isFraud"
]

for col in numeric_columns:
    if train[col].isna().any():
        median_value = train[col].median()

        if pd.isna(median_value):
            median_value = 0

        train[col] = train[col].fillna(median_value)

# Final safety fill.
train.fillna(0, inplace=True)

# Downcast.
for col in train.columns:
    if train[col].dtype == "float64":
        train[col] = train[col].astype("float32")
    elif train[col].dtype == "int64":
        train[col] = train[col].astype("int32")

print("Dropped >90% missing columns:", len(high_missing_cols))
print("Encoded categorical columns:", len(categorical_columns))
print("Remaining missing values:", int(train.isna().sum().sum()))
print("Cleaned shape:", train.shape)

In [ ]:
# Chronological train/test split

# Using time order avoids using future transactions to predict earlier ones.
train.sort_values("TransactionDT", inplace=True)
train.reset_index(drop=True, inplace=True)

feature_columns = [
    c for c in train.columns
    if c not in ["isFraud", "TransactionID", "TransactionDT"]
]

split_index = int(len(train) * 0.80)

X_all = train[feature_columns].to_numpy(dtype="float32")
y_all = train["isFraud"].to_numpy(dtype="int8")

X_train = X_all[:split_index]
X_test = X_all[split_index:]

y_train = y_all[:split_index]
y_test = y_all[split_index:]

del X_all, y_all, train
gc.collect()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Train fraud rate:", round(y_train.mean() * 100, 4), "%")
print("Test fraud rate :", round(y_test.mean() * 100, 4), "%")

In [ ]:
# Baseline XGBoost with class weighting

scale_pos_weight = (
    (y_train == 0).sum() /
    (y_train == 1).sum()
)

baseline_model = XGBClassifier(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.7,
    tree_method="hist",
    max_bin=128,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

t0 = time.time()

baseline_model.fit(X_train, y_train)

baseline_probability = baseline_model.predict_proba(X_test)[:, 1]

baseline_roc_auc = roc_auc_score(y_test, baseline_probability)
baseline_pr_auc = average_precision_score(y_test, baseline_probability)

print("Baseline model trained.")
print("Time:", round(time.time() - t0, 2), "seconds")
print("ROC-AUC:", round(baseline_roc_auc, 4))
print("PR-AUC :", round(baseline_pr_auc, 4))

In [ ]:
# Baseline evaluation at the default 0.50 threshold

baseline_prediction = (
    baseline_probability >= 0.50
).astype(int)

print(
    classification_report(
        y_test,
        baseline_prediction,
        target_names=["Legitimate", "Fraudulent"],
        zero_division=0
    )
)

print("Confusion matrix:")
print(confusion_matrix(y_test, baseline_prediction))

In [ ]:
# RandomUnderSampler + SMOTE

# SMOTE is applied ONLY to the training data.
# The test set remains completely untouched.

n_minority = int((y_train == 1).sum())
n_majority = int((y_train == 0).sum())

# Keeps the experiment practical on Google Colab.
under_target = min(100_000, n_majority)

resampling_pipeline = ImbPipeline(
    steps=[
        (
            "under",
            RandomUnderSampler(
                sampling_strategy={
                    0: under_target,
                    1: n_minority
                },
                random_state=RANDOM_STATE
            )
        ),
        (
            "smote",
            SMOTE(
                sampling_strategy=0.5,
                k_neighbors=5,
                random_state=RANDOM_STATE
            )
        )
    ]
)

t0 = time.time()

X_train_smote, y_train_smote = (
    resampling_pipeline.fit_resample(
        X_train,
        y_train
    )
)

print("Resampling completed.")
print("Time:", round(time.time() - t0, 2), "seconds")

print("\nBefore:")
print(pd.Series(y_train).value_counts())

print("\nAfter:")
print(pd.Series(y_train_smote).value_counts())

print(
    "\nResampled fraud rate:",
    round(y_train_smote.mean() * 100, 2),
    "%"
)

In [ ]:
# Train XGBoost using the resampled training data

smote_model = XGBClassifier(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.7,
    tree_method="hist",
    max_bin=128,
    objective="binary:logistic",
    eval_metric="aucpr",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

t0 = time.time()

smote_model.fit(
    X_train_smote,
    y_train_smote
)

smote_probability = smote_model.predict_proba(
    X_test
)[:, 1]

smote_roc_auc = roc_auc_score(
    y_test,
    smote_probability
)

smote_pr_auc = average_precision_score(
    y_test,
    smote_probability
)

print("SMOTE + XGBoost trained.")
print("Time:", round(time.time() - t0, 2), "seconds")
print("ROC-AUC:", round(smote_roc_auc, 4))
print("PR-AUC :", round(smote_pr_auc, 4))

In [ ]:
# Model comparison at threshold 0.50

smote_prediction_05 = (
    smote_probability >= 0.50
).astype(int)

comparison = pd.DataFrame({
    "Model": [
        "Baseline XGBoost",
        "SMOTE + XGBoost"
    ],
    "Precision": [
        precision_score(y_test, baseline_prediction, zero_division=0),
        precision_score(y_test, smote_prediction_05, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, baseline_prediction, zero_division=0),
        recall_score(y_test, smote_prediction_05, zero_division=0)
    ],
    "F1 Score": [
        f1_score(y_test, baseline_prediction, zero_division=0),
        f1_score(y_test, smote_prediction_05, zero_division=0)
    ],
    "ROC-AUC": [
        baseline_roc_auc,
        smote_roc_auc
    ],
    "PR-AUC": [
        baseline_pr_auc,
        smote_pr_auc
    ]
})

display(comparison)

In [ ]:
# ROC curve

fpr_base, tpr_base, _ = roc_curve(
    y_test,
    baseline_probability
)

fpr_smote, tpr_smote, _ = roc_curve(
    y_test,
    smote_probability
)

plt.figure(figsize=(8, 6))

plt.plot(
    fpr_base,
    tpr_base,
    label=f"Baseline XGBoost (AUC={baseline_roc_auc:.3f})"
)

plt.plot(
    fpr_smote,
    tpr_smote,
    label=f"SMOTE + XGBoost (AUC={smote_roc_auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("/content/figs/02_roc_curve.png", dpi=120)
plt.show()

In [ ]:
# Precision-Recall curve

precision_base, recall_base, _ = precision_recall_curve(
    y_test,
    baseline_probability
)

precision_smote, recall_smote, _ = precision_recall_curve(
    y_test,
    smote_probability
)

plt.figure(figsize=(8, 6))

plt.plot(
    recall_base,
    precision_base,
    label=f"Baseline XGBoost (AP={baseline_pr_auc:.3f})"
)

plt.plot(
    recall_smote,
    precision_smote,
    label=f"SMOTE + XGBoost (AP={smote_pr_auc:.3f})"
)

plt.axhline(
    y_test.mean(),
    linestyle="--",
    label=f"No-skill baseline ({y_test.mean():.3f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("/content/figs/03_precision_recall_curve.png", dpi=120)
plt.show()

In [ ]:
# Decision threshold tuning

thresholds = np.linspace(0.01, 0.99, 99)
threshold_results = []

for threshold in thresholds:

    prediction = (
        smote_probability >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_test, prediction, zero_division=0
        ),
        "Recall": recall_score(
            y_test, prediction, zero_division=0
        ),
        "F1": f1_score(
            y_test, prediction, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

best_index = threshold_df["F1"].idxmax()
best_threshold = float(
    threshold_df.loc[best_index, "Threshold"]
)

print("F1-optimal threshold:", round(best_threshold, 2))

display(
    threshold_df
    .sort_values("F1", ascending=False)
    .head(10)
)

In [ ]:
# Plot threshold trade-off

plt.figure(figsize=(9, 6))

plt.plot(
    threshold_df["Threshold"],
    threshold_df["Precision"],
    label="Precision"
)

plt.plot(
    threshold_df["Threshold"],
    threshold_df["Recall"],
    label="Recall"
)

plt.plot(
    threshold_df["Threshold"],
    threshold_df["F1"],
    label="F1 Score"
)

plt.axvline(
    0.50,
    linestyle=":",
    label="Default = 0.50"
)

plt.axvline(
    best_threshold,
    linestyle="--",
    label=f"F1-optimal = {best_threshold:.2f}"
)

plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Decision Threshold Tuning")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("/content/figs/04_threshold_tuning.png", dpi=120)
plt.show()

In [ ]:
# Final evaluation using the tuned threshold

final_prediction = (
    smote_probability >= best_threshold
).astype(int)

final_precision = precision_score(
    y_test,
    final_prediction,
    zero_division=0
)

final_recall = recall_score(
    y_test,
    final_prediction,
    zero_division=0
)

final_f1 = f1_score(
    y_test,
    final_prediction,
    zero_division=0
)

print("Selected threshold:", round(best_threshold, 2))
print("Precision:", round(final_precision, 4))
print("Recall   :", round(final_recall, 4))
print("F1 Score :", round(final_f1, 4))

print("\nClassification report:")
print(
    classification_report(
        y_test,
        final_prediction,
        target_names=["Legitimate", "Fraudulent"],
        zero_division=0
    )
)

In [ ]:
# Confusion matrices

def plot_confusion(prediction, title, filename):
    cm = confusion_matrix(y_test, prediction)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Legitimate", "Fraudulent"]
    )

    disp.plot()
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filename, dpi=120)
    plt.show()

    tn, fp, fn, tp = cm.ravel()

    print("TN:", tn)
    print("FP:", fp)
    print("FN:", fn)
    print("TP:", tp)


plot_confusion(
    smote_prediction_05,
    "SMOTE + XGBoost — Threshold 0.50",
    "/content/figs/05_confusion_matrix_050.png"
)

plot_confusion(
    final_prediction,
    f"SMOTE + XGBoost — Tuned Threshold {best_threshold:.2f}",
    "/content/figs/06_confusion_matrix_tuned.png"
)

In [ ]:
# XGBoost feature importance

importance_df = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": smote_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

print("Top 20 features:")
display(importance_df.head(20))

In [ ]:
# Plot top 20 features

top_features = importance_df.head(20).sort_values(
    "Importance",
    ascending=True
)

plt.figure(figsize=(10, 8))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 20 XGBoost Feature Importances")

plt.tight_layout()
plt.savefig(
    "/content/figs/07_feature_importance.png",
    dpi=120
)
plt.show()

In [ ]:
# XGBoost gain importance

booster = smote_model.get_booster()

gain_scores = booster.get_score(
    importance_type="gain"
)

gain_sorted = sorted(
    gain_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

gain_df = pd.DataFrame(
    gain_sorted[:20],
    columns=["Feature", "Gain"]
)

display(gain_df)

In [ ]:
# SHAP interpretation

import shap

sample_size = min(3000, len(X_test))

rng = np.random.default_rng(RANDOM_STATE)

sample_indices = rng.choice(
    len(X_test),
    size=sample_size,
    replace=False
)

X_shap = X_test[sample_indices]

explainer = shap.TreeExplainer(
    smote_model
)

shap_values = explainer.shap_values(
    X_shap
)

shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=feature_columns,
    max_display=20
)

In [ ]:
# SHAP feature ranking

mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Mean_Absolute_SHAP": mean_abs_shap
}).sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
)

display(shap_importance.head(20))

In [ ]:
# Final results table

final_summary = pd.DataFrame({
    "Model": [
        "Baseline XGBoost",
        "SMOTE + XGBoost (0.50)",
        f"SMOTE + XGBoost ({best_threshold:.2f})"
    ],
    "Precision": [
        precision_score(y_test, baseline_prediction, zero_division=0),
        precision_score(y_test, smote_prediction_05, zero_division=0),
        final_precision
    ],
    "Recall": [
        recall_score(y_test, baseline_prediction, zero_division=0),
        recall_score(y_test, smote_prediction_05, zero_division=0),
        final_recall
    ],
    "F1 Score": [
        f1_score(y_test, baseline_prediction, zero_division=0),
        f1_score(y_test, smote_prediction_05, zero_division=0),
        final_f1
    ],
    "ROC-AUC": [
        baseline_roc_auc,
        smote_roc_auc,
        smote_roc_auc
    ],
    "PR-AUC": [
        baseline_pr_auc,
        smote_pr_auc,
        smote_pr_auc
    ]
})

display(final_summary)

In [ ]:
# Save outputs for the GitHub project

final_summary.to_csv(
    "/content/data/model_comparison.csv",
    index=False
)

threshold_df.to_csv(
    "/content/data/threshold_results.csv",
    index=False
)

importance_df.to_csv(
    "/content/data/feature_importance.csv",
    index=False
)

gain_df.to_csv(
    "/content/data/gain_importance_top20.csv",
    index=False
)

shap_importance.head(20).to_csv(
    "/content/data/shap_importance_top20.csv",
    index=False
)

with open("/content/data/results.json", "w") as f:
    json.dump({
        "baseline_roc_auc": float(baseline_roc_auc),
        "baseline_pr_auc": float(baseline_pr_auc),
        "smote_roc_auc": float(smote_roc_auc),
        "smote_pr_auc": float(smote_pr_auc),
        "best_threshold": float(best_threshold),
        "final_precision": float(final_precision),
        "final_recall": float(final_recall),
        "final_f1": float(final_f1)
    }, f, indent=2)

print("Saved:")
print("/content/data/model_comparison.csv")
print("/content/data/threshold_results.csv")
print("/content/data/feature_importance.csv")
print("/content/data/gain_importance_top20.csv")
print("/content/data/shap_importance_top20.csv")
print("/content/data/results.json")
print("/content/figs/")